# Fine-tuned model evaluation (baseline + full-strip)

One notebook, two conditions via the `ABLATION` switch:
- `ABLATION = False` -> **baseline** prompts (keep `Reference function: F`), compare to Table 2.
- `ABLATION = True`  -> **full-strip** prompts (strip `F`), compare to the ablation table.

Loads each fine-tuned model as **base + LoRA adapter** and runs the exact same N-way K-shot
protocol as the original inference scripts, so results drop straight into the comparison.

**Protocol (unchanged):** 8-way K-shot stratified sampling · K in {0,1,3,5,10,11} · seeds 2/4/6/8/10
· `USE_SAMPLING=(K==0)` so 0-shot samples (temp 0.7) and 1/3/5/10/11-shot are greedy · `ref_function`
recorded in every output for the scorer.

**Run order:** pick `ABLATION` and `CURRENT_K_SHOT`, run the pipeline cell; repeat for each K, then
flip `ABLATION` and repeat. 2 conditions x 6 shots = 12 pipeline runs per model.


## 1. Imports + login

In [ ]:
import json
import torch
import random
import copy
import gc
import re
from collections import defaultdict
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel                      # load LoRA adapter
from tqdm.notebook import tqdm
from huggingface_hub import notebook_login
notebook_login()

## 2. Config
`MODEL_REGISTRY` maps each fine-tuned model to its base id + adapter dir. Set `ABLATION` and `MODELS_TO_EVAL`. Train-set file is the few-shot **pool** (the de-leaked 1163-row corpus is fine for this; the original pool also works since few-shot demos are drawn from train, never from test).

In [ ]:
MODEL_REGISTRY = {
    "qwen05": dict(
        base="Qwen/Qwen2.5-Coder-0.5B-Instruct",
        # the adapter path is built from rank and train_seed (see ADAPTER_TEMPLATE below)
    ),
    "gemma3": dict(
        base="google/gemma-3-1b-it",
    ),
}
MODELS_TO_EVAL = ["gemma3"]

# rank x training-seed combinations to evaluate. Adapter directories follow the training notebook's naming:
#   out_{MODEL_KEY}_r{rank}_seed{train_seed}_fullstrip/adapter
RANKS_TO_EVAL   = [224]  # thesis Table 10: gemma3 r=224, qwen05 r=16
TRAIN_SEEDS_TO_EVAL = [42, 123, 2024]  # the three training seeds of the thesis
ADAPTER_TEMPLATE = "out_{model_key}_r{rank}_seed{train_seed}_fullstrip/adapter"

TRAIN_SET_FILE = "../data/cleaned/train_only.jsonl"
TEST_SET_FILE  = "../data/cleaned/test_set_150_new.jsonl"

ABLATION = False

## 3. Reference-function stripping (verbatim from ablation script)

In [ ]:
def strip_reference_function(content):
    """Remove the 'Reference function: X' line; return (stripped_content, F)."""
    F = None
    kept = []
    for line in content.split('\n'):
        m = re.match(r'\s*Reference function:\s*(.*?)\s*$', line)
        if m:
            F = m.group(1).strip()
            continue
        kept.append(line)
    while kept and kept[-1].strip() == '':
        kept.pop()
    return '\n'.join(kept), F

## 4. Load + categorize few-shot pool (verbatim)

In [ ]:
def load_and_categorize_data(filepath):
    categorized_data = defaultdict(list)
    print(f"Loading few-shot pool from {filepath}...")
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            data = json.loads(line)
            user_content = ""
            for msg in data.get('messages', []):
                if msg.get('role') == 'user':
                    user_content = msg.get('content', ''); break
            dimension = "Unknown"
            for text_line in user_content.split('\n'):
                if text_line.startswith('Dimension:'):
                    dimension = text_line.replace('Dimension:', '').strip(); break
            pair = [m for m in data.get('messages', []) if m['role'] in ['user', 'assistant']]
            if len(pair) == 2:
                categorized_data[dimension].append(pair)
    print("Categories:")
    for dim, items in categorized_data.items():
        print(f"  - {dim}: {len(items)} items")
    return categorized_data

## 5. N-way K-shot sampling (full-strip applied only when ABLATION=True) — verbatim

In [ ]:
def generate_n_way_k_shot_prompt(categorized_data, K):
    sampled_pairs = []
    for dimension, examples in categorized_data.items():
        actual_k = min(K, len(examples))
        sampled = random.sample(examples, actual_k)
        sampled_pairs.extend(sampled)
    random.shuffle(sampled_pairs)
    flat_few_shot_context = []
    for pair in sampled_pairs:
        if ABLATION:
            pair = copy.deepcopy(pair)
            for msg in pair:
                if msg['role'] == 'user':
                    stripped, _ = strip_reference_function(msg['content'])
                    msg['content'] = stripped
        flat_few_shot_context.extend(pair)
    return flat_few_shot_context

## 6. Inference core — base + LoRA; baseline/full-strip via ABLATION
Only additions vs the originals: `PeftModel.from_pretrained(adapter)` and a `finetuned_` filename prefix. `ref_function` is recorded in **both** conditions so the scorer can map ground truth. (Baseline prompts keep `F`; full-strip prompts have it stripped — the scorer maps on prompt text, so keep baseline/full-strip output files in separate folders when scoring.)

In [ ]:
def generated(MODEL_KEY, rank, train_seed, shared_few_shot_context, K_SHOT=1, seed=42,
              do_sample=False, temperature=0.7, top_p=0.9):
    base_id = MODEL_REGISTRY[MODEL_KEY]["base"]
    adapter = ADAPTER_TEMPLATE.format(model_key=MODEL_KEY, rank=rank, train_seed=train_seed)

    prefix = "finetuned_ablation" if ABLATION else "finetuned_baseline"
    # rank and train_seed in the output file name keep every run's file separate and traceable
    OUTPUT_FILE = (f"{prefix}_{base_id.split('/')[-1].replace('.', '_').lower()}"
                   f"_r{rank}_trainseed{train_seed}_{K_SHOT}shot_seed{seed}.jsonl")

    SYSTEM_PROMPT = "You are a Strudel coding assistant. Write a Strudel code snippet for the following task. Return ONLY valid Strudel code, no markdown, no explanations."

    print(f"\nLoading base: {base_id} + adapter: {adapter}")
    tokenizer = AutoTokenizer.from_pretrained(base_id)
    model = AutoModelForCausalLM.from_pretrained(
        base_id, torch_dtype=torch.bfloat16, device_map="auto")
    model = PeftModel.from_pretrained(model, adapter)
    model.eval()

    with open(TEST_SET_FILE, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    with open(OUTPUT_FILE, 'w', encoding='utf-8') as out_f:
        for line in tqdm(lines, desc=f"r{rank} ts{train_seed} {K_SHOT}shot gs{seed}"):
            if not line.strip(): continue
            data = json.loads(line)
            user_task = data['messages'][1]['content']

            if ABLATION:
                user_task, ref_function = strip_reference_function(user_task)
            else:
                _, ref_function = strip_reference_function(user_task)

            messages_context = copy.deepcopy(shared_few_shot_context)

            if "gemma-2" in base_id.lower():
                if len(messages_context) > 0:
                    messages_context[0]["content"] = f"{SYSTEM_PROMPT}\n\n{messages_context[0]['content']}"
                    messages_context.append({"role": "user", "content": f"Task: {user_task}"})
                    messages = messages_context
                else:
                    messages = [{"role": "user", "content": f"{SYSTEM_PROMPT}\n\nTask: {user_task}"}]
            else:
                messages = [{"role": "system", "content": SYSTEM_PROMPT}]
                messages.extend(messages_context)
                messages.append({"role": "user", "content": user_task})

            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True)
            model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

            torch.manual_seed(seed)

            gen_kwargs = dict(
                max_new_tokens=256,
                pad_token_id=tokenizer.eos_token_id,
                do_sample=do_sample,
            )
            if do_sample:
                gen_kwargs.update(temperature=temperature, top_p=top_p)

            with torch.inference_mode():
                generated_ids = model.generate(**model_inputs, **gen_kwargs)

            generated_ids = [
                out[len(inp):] for inp, out in zip(model_inputs.input_ids, generated_ids)]
            response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
            clean_code = (response.replace("```javascript", "").replace("```js", "")
                          .replace("```strudel", "").replace("```", "").strip())

            result_data = {
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_task},
                    {"role": "assistant", "content": clean_code}
                ],
                "ref_function": ref_function,
                "rank": rank,
                "train_seed": train_seed,
            }
            out_f.write(json.dumps(result_data, ensure_ascii=False) + "\n")

    print(f"Saved: {OUTPUT_FILE}")
    del model; del tokenizer; gc.collect(); torch.cuda.empty_cache()

## 7. Run pipeline
Set `CURRENT_K_SHOT` to one of {0,1,3,5,10,11} and run. Repeat for each K. Then flip `ABLATION` in the config cell and repeat the whole sweep. `USE_SAMPLING=(K==0)` matches the original runs.

In [ ]:
import os

global_categorized_data = load_and_categorize_data(TRAIN_SET_FILE)
SHOT_LIST = [0,1,3,5,10]
SEED_LIST = [2,4,6,8,10]      # generation seeds (for the standard deviation)

done_count = 0
total = len(RANKS_TO_EVAL) * len(TRAIN_SEEDS_TO_EVAL) * len(SHOT_LIST) * len(SEED_LIST) * len(MODELS_TO_EVAL)

for rank in RANKS_TO_EVAL:
    for train_seed in TRAIN_SEEDS_TO_EVAL:
        # Check that the adapter directory exists before starting; if not, skip it and say so,
        # rather than failing halfway because a training run is missing or misnamed
        adapter_dir = ADAPTER_TEMPLATE.format(model_key=MODELS_TO_EVAL[0], rank=rank, train_seed=train_seed)
        if not os.path.isdir(adapter_dir):
            print(f"!! SKIP: adapter not found: {adapter_dir}  (rank={rank}, train_seed={train_seed})")
            continue

        for CURRENT_K_SHOT in SHOT_LIST:
            USE_SAMPLING = (CURRENT_K_SHOT == 0)
            for current_seed in SEED_LIST:
                print("\n" + "="*60)
                print(f"rank={rank} train_seed={train_seed} K={CURRENT_K_SHOT} "
                      f"gen_seed={current_seed} ABLATION={ABLATION}  [{done_count+1}/{total}]")
                print("="*60)

                random.seed(current_seed)
                shared_few_shot_context = generate_n_way_k_shot_prompt(global_categorized_data, K=CURRENT_K_SHOT)

                for MODEL_KEY in MODELS_TO_EVAL:
                    generated(MODEL_KEY, rank, train_seed, shared_few_shot_context,
                              K_SHOT=CURRENT_K_SHOT, seed=current_seed,
                              do_sample=USE_SAMPLING, temperature=0.7, top_p=0.9)
                    done_count += 1

print(f"\nALL DONE — {done_count}/{total} files generated")